In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import T5Tokenizer, T5ForConditionalGeneration
from sklearn.metrics import accuracy_score

In [ ]:
## LOAD DATA

class Data(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        item = self.examples.iloc[idx]
        label = "yes" if item["label"] == 1 else "no"
        input = item["input"]
        return input, label

v_standard_df = pd.read_csv('path/to/V_standard.csv') # load V_standard
v_standard = DataLoader(Data(v_standard_df), batch_size=256) # non-random batch as not needed for eval

v_paraphrase_df = pd.read_csv('path/to/V_paraphrase.csv') # load V_paraphrase
v_paraphrase = DataLoader(Data(v_paraphrase_df), batch_size=256) # non-random batch

In [ ]:
## LOAD MODEL

model_name = "google/flan-t5-large"
tokeniser = T5Tokenizer.from_pretrained(model_name)
model_loc = "path/to/model/weights"
model = T5ForConditionalGeneration.from_pretrained(model_loc, # model_name for TS0
                                                   torch_dtype=torch.bfloat16,
                                                   device_map='cuda')

In [ ]:
# function to calculate accuracies
def get_metrics(preds, targets):
    overall_acc = accuracy_score(targets, preds)
    positive_acc = sum(1 for i in range(len(preds)) if preds[i] == targets[i] and targets[i] == 1) / len([i for i in targets if i == 1])
    negative_acc = sum(1 for i in range(len(preds)) if preds[i] == targets[i] and targets[i] == 0) / len([i for i in targets if i == 0])
    return overall_acc, positive_acc, negative_acc

# function to evaluate model
def model_eval(dataset, dataset_df):
    preds = []
    model.eval() # model in eval() mode
    with torch.no_grad():
        for batch in tqdm(dataset):
            texts, labels = batch

            inputs = tokeniser(texts, return_tensors="pt", truncation=True, padding=True, max_length=512).to("cuda") # tokenise texts
            labels = tokeniser(labels, return_tensors="pt", truncation=True, padding=True, max_length=2).input_ids.to("cuda") # tokenise labels

            answer_output = model.generate(**inputs, max_length=2, use_cache=False) # get model outputs
            batch_predictions = [
                1 if "yes" == tokeniser.decode(output, skip_special_tokens=True).lower() else 0 for output in answer_output # convert raw token outputs to binary
            ]

            preds.extend(batch_predictions)

    overall_acc, positive_acc, negative_acc = get_metrics(preds, dataset_df['label'].to_list()) # evaluate accuracies

    return positive_acc, negative_acc, overall_acc

In [ ]:
print()
positive_acc, negative_acc, overall_acc = model_eval(v_standard, v_standard_df)
print(f"{positive_acc:.3f} & {negative_acc:.3f} & {overall_acc:.3f}")
print()
positive_acc, negative_acc, overall_acc = model_eval(v_paraphrase, v_paraphrase_df)
print(f"{positive_acc:.3f} & {negative_acc:.3f} & {overall_acc:.3f}")